In [ ]:
# from pathlib import Path

# file = Path.home() / "Downloads"

# for f in file.iterdir():
#     print(f, f.suffix)




In [21]:
from pathlib import Path
import pandas as pd

# ==========================
# PATHS
# ==========================
OUT = Path("flight_data")
OUT.mkdir(exist_ok=True)

DATA_DIR = Path.home() / "Downloads"
FILE_PATH = DATA_DIR / "On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2025_1.csv"

output_file = OUT / "airport_hourly_flights_2025_01.csv"

AIRPORTS = [
    "TPA","STL","SNA","SMF","SLC","SJC","SFO","SEA","SAT","SAN","RSW","RDU","PIT",
    "PHX","PHL","PDX","ORD","OGG","OAK","MSY","MSP","MIA","MDW","MCO","MCI","LGA",
    "LAX","LAS","JFK","IND","IAH","IAD","HOU","HNL","FLL","EWR","DTW","DFW","DEN",
    "DCA","DAL","CVG","CMH","CLT","CLE","BWI","BOS","BNA","AUS","ATL"
]

# ==========================
# LOAD
# ==========================
def load_data():
    print(f"Loading: {FILE_PATH}")

    df = pd.read_csv(FILE_PATH, low_memory=False)
    df.columns = df.columns.str.strip()

    return df

# ==========================
# PROCESS
# ==========================
def process(df):

    # ensure types
    df["CRSDepTime"] = df["CRSDepTime"].fillna(0).astype(int)
    df["Hour"] = df["CRSDepTime"] // 100

    # ensure valid rows only
    df = df.dropna(subset=["Origin", "Dest", "FlightDate"])

    # ==========================
    # FILTER SEPARATELY (IMPORTANT FIX)
    # ==========================

    dep_df = df[df["Origin"].isin(AIRPORTS)]
    arr_df = df[df["Dest"].isin(AIRPORTS)]

    # ==========================
    # DEPARTURES
    # ==========================
    dep = (
        dep_df.groupby(["FlightDate", "Origin", "Hour"])
        .size()
        .reset_index(name="Departures")
        .rename(columns={"Origin": "Airport"})
    )

    # ==========================
    # ARRIVALS
    # ==========================
    arr = (
        arr_df.groupby(["FlightDate", "Dest", "Hour"])
        .size()
        .reset_index(name="Arrivals")
        .rename(columns={"Dest": "Airport"})
    )

    # ==========================
    # MERGE
    # ==========================
    merged = pd.merge(
        dep,
        arr,
        on=["FlightDate", "Airport", "Hour"],
        how="outer"
    ).fillna(0)

    merged["Departures"] = merged["Departures"].astype(int)
    merged["Arrivals"] = merged["Arrivals"].astype(int)

    merged["TotalFlights"] = merged["Departures"] + merged["Arrivals"]

    return merged

# ==========================
# RUN
# ==========================
df = load_data()
out = process(df)

out.to_csv(output_file, index=False)

print("Saved:", output_file)
print(out.head())

Loading: C:\Users\alecg\Downloads\On_Time_Reporting_Carrier_On_Time_Performance_(1987_present)_2025_1.csv
Saved: flight_data\airport_hourly_flights_2025_01.csv
   FlightDate Airport  Hour  Departures  Arrivals  TotalFlights
0  2025-01-01     ATL     2           0         1             1
1  2025-01-01     ATL     5           5        28            33
2  2025-01-01     ATL     6          11        96           107
3  2025-01-01     ATL     7          22        55            77
4  2025-01-01     ATL     8          51        40            91
